**Table of contents**<a id='toc0_'></a>    
- 1. [RAGFlow本地部署](#toc1_)    
  - 1.1. [前提条件](#toc1_1_)    
  - 1.2. [启动服务器](#toc1_2_)    
    - 1.2.1. [确保 vm.max_map_count 不小于 262144：](#toc1_2_1_)    
    - 1.2.2. [ 克隆ragflow源码](#toc1_2_2_)    
    - 1.2.3. [ Docker 镜像启动服务器](#toc1_2_3_)    
    - 1.2.4. [把Dify停掉](#toc1_2_4_)    
    - 1.2.5. [再次确认服务器状态](#toc1_2_5_)    
- 2. [RAGFlow模型配置](#toc2_)    
  - 2.1. [添加模型](#toc2_1_)    
    - 2.1.1. [添加Ollama模型](#toc2_1_1_)    
- 3. [RAGFlow创建知识库](#toc3_)    
- 4. [RAGFlow API调用](#toc4_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[RAGFlow本地部署](#toc0_)

## 1.1. <a id='toc1_1_'></a>[前提条件](#toc0_)

* CPU >= 4 核
* RAM >= 16 GB
* Disk >= 50 GB
* Docker >= 24.0.0 & Docker Compose >= v2.26.1  
如果你并没有在本机安装 Docker（Windows、Mac，或者 Linux）, 可以参考文档Docker Engine 自行安装。

## 1.2. <a id='toc1_2_'></a>[启动服务器](#toc0_)

### 1.2.1. <a id='toc1_2_1_'></a>[确保 vm.max_map_count 不小于 262144：](#toc0_)
如需确认 vm.max_map_count 的大小：  
$ sysctl vm.max_map_count 

<img src="./Image/2025-06-02-21-22-30.png" style="margin-left: 0" width="50%">

 
如果 vm.max_map_count 的值小于 262144，可以进行重置：  
这里我们设为 262144:    
$ sudo sysctl -w vm.max_map_count=262144 

 你的改动会在下次系统重启时被重置。如果希望做永久改动，还需要在 /etc/sysctl.conf 文件里把 vm.max_map_count 的值再相应更新一遍：vm.max_map_count=262144，设置方法如下：  
1. sudo vim /etc/sysctl.conf
2. 滑到文末，按下Ins按键，粘贴 vm.max_map_count=262144 ，按 ESC 退出，在按shift+z+z保存。

<img src="./Image/2025-06-02-21-31-38.png" style="margin-left: 0" width="60%">

3. 通过 sudo sysctl -p 使其生效  
(base) root@nishuixingzhou:~# sudo sysctl -p  
输出：  vm.max_map_count = 262144

4. 通过 sysctl vm.max_map_count 确认是否修改成功，输出：  
(base) root@nishuixingzhou:~# sysctl vm.max_map_count  
vm.max_map_count = 262144


### 1.2.2. <a id='toc1_2_2_'></a>[ 克隆ragflow源码](#toc0_)

[ragflow官网链接](https://github.com/infiniflow/ragflow#)

git clone https://github.com/infiniflow/ragflow.git

### 1.2.3. <a id='toc1_2_3_'></a>[ Docker 镜像启动服务器](#toc0_)

进入 docker 文件夹，利用提前编译好的 Docker 镜像启动服务器  
请注意，目前官方提供的所有 Docker 镜像均基于 x86 架构构建，并不提供基于 ARM64 的 Docker 镜像。 如果你的操作系统是 ARM64 架构，请参考[Build RAGFlow Docker image](https://ragflow.io/docs/dev/build_docker_image)
这篇文档自行构建 Docker 镜像。

如果是Docker Desktop，要先运行Docker Desktop，否会报错。  
(base) root@nishuixingzhou:~/AI-WSL/soft/ragflow-main/docker# sudo docker compose -f docker-compose-gpu.yml up -d  
sudo: docker: command not found

### 1.2.4. <a id='toc1_2_4_'></a>[把Dify停掉](#toc0_)

还要注意一个问题，RAGFlow用的端口是80，Dify用的端口也是80，如果没有80端口转发，就手动添加一个，运行RAGFlow的时候要把Dify停掉。9380是它提供的服务端口、443是存储数据的端口。防火墙要把9380和80以及443这三个端口放开。把Dify停掉要通过docker-compose down命令停止，否则可能会自启动。

(base) root@nishuixingzhou:~# cd /root/AI-WSL/project/projectDify_DeepSeek/code/dify-main/docker

(envollama) root@nishuixingzhou:~/AI-WSL/project/projectDify_DeepSeek/code/dify-main/docker# sudo docker-compose down

<img src="./Image/2025-06-02-22-20-19.png" style="margin-left: 0" width="70%">

运行以下命令会自动下载 RAGFlow slim Docker 镜像v0.18.0-slim 。请参考下表查看不同Docker 发行版的描述。如需下载不同于v0.18.0-slim 的 Docker 镜像，请在运行docker compose 启动服务之前先更新 docker/.env 文件内的RAGFLOW_IMAGE 变量。比如，你可以通过设置RAGFLOW_IMAGE=infiniflow/ragflow:v0.18.0 来下载 RAGFlow 镜像的 
v0.18.0 完整发行版。

$cd ragflow/docker

#Use CPU for embedding and DeepDoc tasks:  
#$ docker compose -f docker-compose.yml up -d

#To use GPU to accelerate embedding and DeepDoc tasks:  
sudo docker compose -f docker-compose-gpu.yml up -d

例如：  
(base) root@nishuixingzhou:~# cd /root/AI-WSL/soft/ragflow-main/docker  
(base) root@nishuixingzhou:~/AI-WSL/soft/ragflow-main/docker# sudo docker compose -f docker-compose-gpu.yml up -d  

<img src="./Image/2025-06-03-00-55-04.png" style="margin-left: 0" width="70%">

如果你遇到 Docker 镜像拉不下来的问题，可以在 docker/.env 文件内根据变量RAGFLOW_IMAGE 的注释提示选择华为云或者阿里云的相应镜像。

* 华为云镜像名：  
swr.cn-north-4.myhuaweicloud.com/infiniflow/ragflow
* 阿里云镜像名：  
registry.cn-hangzhou.aliyuncs.com/infiniflow/ragflow

### 1.2.5. <a id='toc1_2_5_'></a>[再次确认服务器状态](#toc0_)

服务器启动成功后再次确认服务器状态

docker logs -f ragflow-server

出现以下界面提示说明服务器启动成功：

<img src="./Image/2025-06-03-01-01-18.png" style="margin-left: 0" width="80%">

如果您在没有看到上面的提示信息出来之前，就尝试登录 RAGFlow，你的浏览器有可能会提示 network anormal 或 网络异常。在你的浏览器中输入你的服务器对应的 IP 地址并登录 RAGFlow。上面这个例子中，您只需输入http://IP_OF_YOUR_MACHINE 即可：未改动过配置则无需输入端口（默认的 HTTP 服务端口 80）。


<img src="./Image/2025-06-03-01-06-22.png" style="margin-left: 0" width="80%">

下面注册登录就行。

<img src="./Image/2025-06-03-01-09-56.png" style="margin-left: 0" width="80%">

# 2. <a id='toc2_'></a>[RAGFlow模型配置](#toc0_)

## 2.1. <a id='toc2_1_'></a>[添加模型](#toc0_)

### 2.1.1. <a id='toc2_1_1_'></a>[添加Ollama模型](#toc0_)

添加模型方法和Dify类似

<img src="./Image/2025-06-03-01-27-28.png" style="margin-left: 0" width="80%">

ollama list查询ollama模型名称

(base) root@nishuixingzhou:~/AI-WSL/project/projectDify_DeepSeek/code/dify-main/docker# ollama list  
NAME                                          ID              SIZE      MODIFIED     
nn200433/text2vec-bge-large-chinese:latest    5907030ba704    207 MB    25 hours ago    
deepseek-r1:1.5b                              e0979632db5a    1.1 GB    2 days ago      
qwen2.5_1.5B-Instruct-gguf:latest             5c0da8648104    3.1 GB    6 weeks ago     
qwen2.5_1.5B-Instruct-merged:latest           44f87b5f3053    3.1 GB    6 weeks ago 

以deepseek-r1:1.5b模型为例配置

如果是远程服务器，“基础 Url”需要用 本地物理机的实际IP地址+端口号（默认为11434），但是和dify配置时候同理，本地windows，用docker部署了RAGFlow，没有用docker部署ollama。

Docker进程和主机服务进行通信，这里面存在映射关系。需要改为：http://host.docker.internal:11434

host.docker.internal:
这是 Docker 提供的一个特殊主机名。当在 Docker 容器内部使用时，它会解析为主机机器上 Docker 守护进程的 IP 地址。它允许容器与主机机器上运行的服务进行通信。

正确和错误配置方式如下：

<img src="./Image/2025-06-03-01-39-20.png" style="margin-left: 0" width="35%">  <img src="./Image/2025-06-03-01-42-58.png" style="margin-left: 0" width="55%">

添加成功

<img src="./Image/2025-06-03-01-46-12.png" style="margin-left: 0" width="80%">

上面配置的是chat模型，和Dify不同的是，RAGFlow只配chat模型是不能使用的，必须要把embedding model也配置了才行。否则新建助理的时候会报错：

<img src="./Image/2025-06-03-01-52-21.png" style="margin-left: 0" width="80%">

继续添加embedding model模型

<img src="./Image/2025-06-03-01-53-24.png" style="margin-left: 0" width="80%">

* 模型类型：embedding
* 模型名称：ollama list查询出来的名称 nn200433/text2vec-bge-large-chinese:latest
* 基础Url：http://host.docker.internal:11434
* 最大token数：这款nn200433/text2vec-bge-large-chinese:latest模型的最大token数是1024，这个是根据模型来设置的，比如deepseek-r1:1.5b作为embedding model可以设置为4096。

<img src="./Image/2025-06-03-01-56-14.png" style="margin-left: 0" width="40%">

<img src="./Image/2025-06-03-02-11-39.png" style="margin-left: 0" width="70%">

继续设置默认模型

![](Image/2025-06-03-02-13-20.png)

# 3. <a id='toc3_'></a>[RAGFlow创建知识库](#toc0_)


# 4. <a id='toc4_'></a>[RAGFlow API调用](#toc0_)

视频进度：02:00:56